# DATA 2001 Group Assignment 2026

**Topic:** NSW regional statistics, Greater Sydney POI data, SA2 resource scoring, and report analysis.

**Due:** 20 May 2026, 11:59 PM

This notebook is structured to document the full workflow required for the group submission.

## Deliverables Checklist

- PDF report
- Jupyter Notebook describing the full workflow
- Tutor conversation in Week 12 or Week 13
- One group ZIP file submitted to Canvas

## Setup

In [ ]:
from pathlib import Path
from tabulate import tabulate
import json
import math
import sqlite3
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd

DATA_DIR = Path.cwd()
CSV_PATH = DATA_DIR / "Region summary_ New South Wales STE 1.csv"
DB_PATH = DATA_DIR / "data2001_assignment.sqlite"

CSV_PATH

# Task 1: NSW Summary Statistics

## 1.1 Load the NSW Region Summary CSV

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
df_raw.head()

In [ ]:
print("Shape:", df_raw.shape)
display(df_raw.info())
display(df_raw.describe(include="all"))

## 1.2 Data Cleaning

In [ ]:
df_clean = df_raw.copy()

# Standardise column names and text fields.
df_clean.columns = df_clean.columns.str.strip()
text_columns = ["Measure Code", "Parent Description", "Description"]
for column in text_columns:
    df_clean[column] = df_clean[column].astype("string").str.strip()

year_columns = [column for column in df_clean.columns if column.isdigit()]
df_clean[year_columns] = df_clean[year_columns].apply(pd.to_numeric, errors="coerce")

# Remove exact duplicate rows if present.
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print("Cleaned shape:", df_clean.shape)
df_clean.head()

In [ ]:
missing_summary = (
    df_clean.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(df_clean) * 100).round(2)
missing_summary

### Cleaning Notes

- The identifier columns are `Measure Code`, `Parent Description`, and `Description`.
- Year columns from 2011 to 2025 were detected and converted to numeric values.
- Missing values appear because not every statistic is available for every year. For example, 2025 only has limited available data, so most time-based analysis uses 2019-2024 instead.
- Exact duplicate rows were checked and removed if present.
- Text columns were stripped of extra whitespace to make filtering and matching more reliable.
- Numeric year columns were converted with `pd.to_numeric(errors="coerce")`, so invalid values become missing values rather than breaking the workflow.


## 1.3 Derived Statistics

Each group member should contribute 5 derived statistics. Use this section to clearly label each member's work.

In [ ]:
latest_year = max(year_columns, key=int)
usable_years = [column for column in year_columns if df_clean[column].notna().any()]
latest_usable_year = max(usable_years, key=int)

print("All year columns:", year_columns)
print("Latest year column:", latest_year)
print("Latest usable year:", latest_usable_year)

### Ronnie Luo: Derived Statistics

Planned statistics:

1. Child share of population: percentage of NSW residents aged 0-14.
2. Older population share: percentage of NSW residents aged 65 and over.
3. Ageing index: number of residents aged 65+ per 100 children aged 0-14.
4. Age dependency ratio: children and older residents per 100 working-age residents.
5. Population density change: change in persons per square kilometre from 2019 to 2024.

In [ ]:
analysis_years = [str(year) for year in range(2019, 2025)]

def metric_series(description):
    matches = df_clean[df_clean["Description"].eq(description)]
    if matches.empty:
        raise ValueError(f"Metric not found: {description}")
    return matches.iloc[0][analysis_years].astype(float)

def age_group_total(age_labels):
    total = pd.Series(0.0, index=analysis_years)
    for sex in ["Males", "Females"]:
        for age_label in age_labels:
            total += metric_series(f"{sex} - {age_label} (no.)")
    return total

total_population = metric_series("Estimated resident population (no.)")
working_age_population = metric_series("Working age population (aged 15-64 years) (no.)")
working_age_share = metric_series("Working age population (aged 15-64 years) (%)")
population_density = metric_series("Population density (persons/km2)")

children_0_14 = age_group_total(["0-4 years", "5-9 years", "10-14 years"])
older_65_plus = age_group_total(["65-69 years", "70-74 years", "75-79 years", "80-84 years", "85 and over"])

ronnie_stats = pd.DataFrame({
    "year": analysis_years,
    "child_share_pct": (children_0_14 / total_population * 100).round(2).values,
    "older_share_pct": (older_65_plus / total_population * 100).round(2).values,
    "ageing_index_older_per_100_children": (older_65_plus / children_0_14 * 100).round(2).values,
    "dependency_ratio_dependents_per_100_working_age": ((children_0_14 + older_65_plus) / working_age_population * 100).round(2).values,
    "working_age_share_pct": working_age_share.round(2).values,
    "population_density_persons_per_km2": population_density.round(2).values,
})

ronnie_summary = pd.DataFrame({
    "derived_statistic": [
        "Child share of population",
        "Older population share",
        "Ageing index",
        "Age dependency ratio",
        "Population density",
    ],
    "2019_value": [
        ronnie_stats.loc[0, "child_share_pct"],
        ronnie_stats.loc[0, "older_share_pct"],
        ronnie_stats.loc[0, "ageing_index_older_per_100_children"],
        ronnie_stats.loc[0, "dependency_ratio_dependents_per_100_working_age"],
        ronnie_stats.loc[0, "population_density_persons_per_km2"],
    ],
    "2024_value": [
        ronnie_stats.loc[5, "child_share_pct"],
        ronnie_stats.loc[5, "older_share_pct"],
        ronnie_stats.loc[5, "ageing_index_older_per_100_children"],
        ronnie_stats.loc[5, "dependency_ratio_dependents_per_100_working_age"],
        ronnie_stats.loc[5, "population_density_persons_per_km2"],
    ],
})
ronnie_summary["change"] = (ronnie_summary["2024_value"] - ronnie_summary["2019_value"]).round(2)
ronnie_summary["interpretation"] = [
    "Percentage-point change in residents aged 0-14.",
    "Percentage-point change in residents aged 65+.",
    "Change in older residents per 100 children.",
    "Change in dependents per 100 working-age residents.",
    "Change in persons per square kilometre.",
]

display(ronnie_stats)
display(ronnie_summary)

### Spencer: Derived Statistics

Planned statistics:

1. Business Entry and Exit Rate
2. Employee Income Mean/Median Ratio
3. Share of Business Sizes
4. Capital Gains Mean/Median Ratio
5. Wage Growth Rate

In [ ]:
rows = []

for i in range(2021, 2025):
    totals = df_clean[df_clean['Description'] == 'Total number of businesses'][str(i)].values[0]
    exits = df_clean[df_clean['Description'] == 'Total number of business exits'][str(i)].values[0]
    entries = df_clean[df_clean['Description'] == 'Total number of business entries'][str(i)].values[0]

    exit_rate = (exits / totals) * 100
    entry_rate = (entries / totals) * 100

    rows.append([i, f"{int(totals)}", f"{int(exits)}", f"{round(exit_rate, 1)}%", f"{int(entries)}", f"{round(entry_rate, 1)}%"])


print(tabulate(rows, headers=["Year", "Total Businesses", "Exits", "Exit Rate", "Entries", "Entry Rate"], tablefmt="pretty"))

In [ ]:
rows = []

for i in range(2018, 2023):
    median = df_clean[df_clean['Description'] == 'Median employee income ($)'][str(i)].values[0]
    mean = df_clean[df_clean['Description'] == 'Mean employee income ($)'][str(i)].values[0]
    
    ratio = mean / median
    rows.append([i, f"${int(mean)}", f"${int(median)}", f"{round(ratio, 2)}"])

print(tabulate(rows, headers=["Year", "Mean Employee Income ($)", "Median Employee Income ($)", "Mean-Median Ratio"], tablefmt="pretty"))

In [ ]:
rows = []

for i in range(2020, 2025):
    non_employing = df_clean[df_clean['Description'] == 'Number of non-employing businesses'][str(i)].values[0]
    small = df_clean[df_clean['Description'] == 'Number of employing businesses: 1-4 employees'][str(i)].values[0]
    medium = df_clean[df_clean['Description'] == 'Number of employing businesses: 5-19 employees'][str(i)].values[0]
    large = df_clean[df_clean['Description'] == 'Number of employing businesses: 20 or more employees'][str(i)].values[0]

    count = non_employing + small + medium + large

    rows.append([i, f"{round(non_employing/count * 100, 1)}%", f"{round(small/count * 100, 1)}%", f"{round(medium/count * 100, 1)}%", f"{round(100 - round(non_employing/count * 100, 1) - round(small/count * 100, 1) - round(medium/count * 100, 1), 1)}%"])
    
print(tabulate(rows, headers=["Year", "Share of Non-Employing", "Share of Micro (1-4 staff)", "Share of Small (5-19 staff)", "Share of Large (20+ staff)"], tablefmt="pretty"))

In [ ]:
rows = []

for i in range(2017, 2023):
    claimants = int(df_clean[df_clean['Description'] == 'Persons who reported gross capital gains (no.)'][str(i)].values[0])
    median_cg = df_clean[df_clean['Description'] == 'Median value of gross capital gains ($)'][str(i)].values[0]
    mean_cg = df_clean[df_clean['Description'] == 'Mean value of gross capital gains ($)'][str(i)].values[0]

    ratio = mean_cg / median_cg
    rows.append([str(i), str(claimants), f"{int(mean_cg)}", f"{int(median_cg)}", f"{round(ratio, 1)}"])

print(tabulate(rows, headers=["Year", "Claimants", "Mean Capital Gain ($)", "Median Capital Gain ($)", "Mean/Median Ratio"], tablefmt="pretty"))

In [ ]:
rows = []

years = [2018, 2019, 2020, 2021, 2022]

for i in range(1, len(years)):
    prev = df_clean[df_clean['Description'] == 'Median employee income ($)'][str(years[i-1])].values[0]
    curr = df_clean[df_clean['Description'] == 'Median employee income ($)'][str(years[i])].values[0]

    growth = ((curr - prev) / prev) * 100
    rows.append([f"{years[i-1]}-{years[i]}", f"${int(prev):,}", f"${int(curr):,}", f"{round(growth, 1)}%"])

print(tabulate(rows, headers=["Period", "Previous Median ($)", "Current Median ($)", "Wage Growth Rate"], tablefmt="pretty"))

### Arya: Derived Statistics


Planned statistics:

1. Median age of persons change (2019 to 2024)
2. Working-age population growth (2019 to 2024)
3. Working-age share change (2019 to 2024)
4. Female median age minus overall median age in 2024
5. Male median age minus overall median age in 2024

In [ ]:
median_age_2019 = df_clean.loc[df_clean["Description"] == "Median age - persons (years)", "2019"].values[0]
median_age_2024 = df_clean.loc[df_clean["Description"] == "Median age - persons (years)","2024"].values[0]

median_age_change = median_age_2024 - median_age_2019

print(f"Median age in 2019: {median_age_2019:.1f} years")
print(f"Median age in 2024: {median_age_2024:.1f} years")
print(f"Change in median age from 2019 to 2024: {median_age_change:.2f} years")

In [ ]:
working_age_2019 = df_clean.loc[df_clean["Description"] == "Working age population (aged 15-64 years) (no.)", "2019"].values[0]
working_age_2024 = df_clean.loc[df_clean["Description"] == "Working age population (aged 15-64 years) (no.)", "2024"].values[0]
working_age_growth_pct = ((working_age_2024 - working_age_2019) / working_age_2019) * 100

print(f"Working-age population in 2019: {working_age_2019:,.0f}")
print(f"Working-age population in 2024: {working_age_2024:,.0f}")
print(f"Working-age population growth from 2019 to 2024: {working_age_growth_pct:.2f}%")

In [ ]:
working_share_2019 = df_clean.loc[df_clean["Description"] == "Working age population (aged 15-64 years) (%)", "2019"].values[0]
working_share_2024 = df_clean.loc[df_clean["Description"] == "Working age population (aged 15-64 years) (%)", "2024"].values[0]
working_share_change = working_share_2024 - working_share_2019

print(f"Working-age share in 2019: {working_share_2019:.1f}%")
print(f"Working-age share in 2024: {working_share_2024:.1f}%")
print(f"Change in working-age share from 2019 to 2024: {working_share_change:.2f} percentage points")

In [ ]:
female_median_age_2024 = df_clean.loc[df_clean["Description"] == "Median age - females (years)", "2024"].values[0]
overall_median_age_2024 = df_clean.loc[df_clean["Description"] == "Median age - persons (years)", "2024"].values[0]
female_vs_overall_gap = female_median_age_2024 - overall_median_age_2024

print(f"Female median age in 2024: {female_median_age_2024:.1f} years")
print(f"Overall median age in 2024: {overall_median_age_2024:.1f} years")
print(f"Female minus overall median age in 2024: {female_vs_overall_gap:.2f} years")

In [ ]:
male_median_age_2024 = df_clean.loc[df_clean["Description"] == "Median age - males (years)", "2024"].values[0]
overall_median_age_2024 = df_clean.loc[df_clean["Description"] == "Median age - persons (years)", "2024"].values[0]
male_vs_overall_gap = male_median_age_2024 - overall_median_age_2024

print(f"Male median age in 2024: {male_median_age_2024:.1f} years")
print(f"Overall median age in 2024: {overall_median_age_2024:.1f} years")
print(f"Male minus overall median age in 2024: {male_vs_overall_gap:.2f} years")

### Ananya: Derived Statistics


Planned statistics:

1. Statistic 1: Population growth rate (2019 to 2024): NSW's average annual population growth rate over the most data-rich 5-year window in the dataset.
2. Statistic 2: Gender balance ratio over time. Tracks the male-to-female population ratio across each year, revealing demographic shifts over the decade.
3. Statistic 3: Year with the most data coverage bonus candidate. Finds which year has the most non-null values across all 800 measure
4. Statistic 4: Median age gap between males and females bonus candidate. Computes the difference in median age between males and females each year. 
5. Statistic 5:  Breadth of topic coverage across parent categories bonus candidate

In [ ]:
pop_2019 = df_clean.loc[df_clean["Description"] == "Estimated resident population (no.)", "2019"].values[0]
pop_2024 = df_clean.loc[df_clean["Description"] == "Estimated resident population (no.)", "2024"].values[0]

cagr = ((pop_2024 / pop_2019) ** (1/5) - 1) * 100

print(f"NSW population 2019: {pop_2019:,.0f}")
print(f"NSW population 2024: {pop_2024:,.0f}")
print(f"Average annual growth rate (CAGR): {cagr:.2f}%")

In [ ]:
males = df_clean[df_clean["Description"].str.contains("males \(no\.\)", na=False, regex=True)]
females = df_clean[df_clean["Description"].str.contains("females \(no\.\)", na=False, regex=True)]

year_cols = [c for c in df_clean.columns if c.isdigit()]

gender_ratio = pd.DataFrame({
    "year": year_cols,
    "males": males[year_cols].values[0],
    "females": females[year_cols].values[0]
}).dropna()

gender_ratio["male_to_female_ratio"] = (gender_ratio["males"] / gender_ratio["females"]).round(4)

print(gender_ratio[["year", "male_to_female_ratio"]].to_string(index=False))
print(f"\nMost balanced year: {gender_ratio.loc[gender_ratio['male_to_female_ratio'].sub(1).abs().idxmin(), 'year']}")

In [ ]:
year_cols = [c for c in df_clean.columns if c.isdigit()]

coverage = df_clean[year_cols].notna().sum().rename("non_null_count").to_frame()
coverage["coverage_pct"] = (coverage["non_null_count"] / len(df_clean) * 100).round(1)
coverage = coverage.sort_values("coverage_pct", ascending=False)

print("Data coverage by year:")
print(coverage.to_string())
print(f"\nBest year to anchor analysis: {coverage.index[0]} ({coverage['coverage_pct'].iloc[0]}% coverage)")

In [ ]:
male_age = df_clean[df_clean["Description"] == "Median age - males (years)"]
female_age = df_clean[df_clean["Description"] == "Median age - females (years)"]

year_cols = [c for c in df_clean.columns if c.isdigit()]

age_df = pd.DataFrame({
    "year": year_cols,
    "median_age_male": male_age[year_cols].values[0],
    "median_age_female": female_age[year_cols].values[0]
}).dropna()

age_df["age_gap_f_minus_m"] = (age_df["median_age_female"] - age_df["median_age_male"]).round(1)

print(age_df.to_string(index=False))
print(f"\nAverage gap (females older by): {age_df['age_gap_f_minus_m'].mean():.2f} years")
print(f"Trend: {'widening' if age_df['age_gap_f_minus_m'].iloc[-1] > age_df['age_gap_f_minus_m'].iloc[0] else 'narrowing'}")

In [ ]:
year_cols = [c for c in df_clean.columns if c.isdigit()]

category_stats = df_clean.groupby("Parent Description").agg(
    measure_count=("Description", "nunique"),
    avg_coverage_pct=(year_cols[0], lambda x: df_clean.loc[x.index, year_cols].notna().mean(axis=1).mean() * 100)
).round(1).sort_values("measure_count", ascending=False)

print("Top 10 parent categories by number of measures:")
print(category_stats.head(10).to_string())
print(f"\nTotal parent categories: {len(category_stats)}")
print(f"Most data-complete category: {category_stats['avg_coverage_pct'].idxmax()}")

# Task 2: Greater Sydney SA2/SA4 Points of Interest Dataset

## 2.1 Select SA4 Zones

Each group member should select one distinct Greater Sydney SA4 zone.

Record the selected zones here:

- Member 1: SA4 zone name
- Member 2: SA4 zone name
- Member 3: SA4 zone name
- Member 4: SA4 zone name

In [ ]:
SELECTED_SA4_ZONES = [
    # "Sydney - City and Inner South",
    # "Parramatta",
]

# Add or load SA2 boundary data here when available.
# Expected columns: sa4_name, sa2_name, min_lon, min_lat, max_lon, max_lat
sa2_boundaries = pd.DataFrame(columns=["sa4_name", "sa2_name", "min_lon", "min_lat", "max_lon", "max_lat"])
sa2_boundaries

## 2.2 NSW Points of Interest API Function

In [ ]:
def fetch_pois_in_bbox(min_lon, min_lat, max_lon, max_lat, limit=1000):
    """Return POI records inside a bounding box.

    Update API_URL and params after confirming the NSW POI API endpoint from the Week 8 tutorial.
    """
    API_URL = "TODO_ADD_NSW_POI_API_ENDPOINT"
    params = {
        "min_lon": min_lon,
        "min_lat": min_lat,
        "max_lon": max_lon,
        "max_lat": max_lat,
        "limit": limit,
    }

    if API_URL.startswith("TODO"):
        return pd.DataFrame()

    url = API_URL + "?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url) as response:
        payload = json.loads(response.read().decode("utf-8"))

    records = payload.get("features", payload if isinstance(payload, list) else [])
    return pd.json_normalize(records)


## 2.3 Loop Through SA2 Regions

In [ ]:
poi_frames = []

for _, row in sa2_boundaries.iterrows():
    if SELECTED_SA4_ZONES and row["sa4_name"] not in SELECTED_SA4_ZONES:
        continue

    pois = fetch_pois_in_bbox(row["min_lon"], row["min_lat"], row["max_lon"], row["max_lat"])
    if pois.empty:
        continue

    pois["sa4_name"] = row["sa4_name"]
    pois["sa2_name"] = row["sa2_name"]
    poi_frames.append(pois)

pois_df = pd.concat(poi_frames, ignore_index=True) if poi_frames else pd.DataFrame()
print("POI rows collected:", len(pois_df))
pois_df.head()

## 2.4 Store POI Dataset in Local Database

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    if not pois_df.empty:
        pois_df.to_sql("points_of_interest", conn, if_exists="replace", index=False)
        print("Saved points_of_interest table to", DB_PATH)
    else:
        print("No POI data saved yet. Complete the API endpoint and SA2 boundary inputs first.")

# Task 3: SA2 Well-Resourced Score

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def z_score(series):
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(0, index=series.index)
    return (series - series.mean()) / std

if not pois_df.empty and "sa2_name" in pois_df.columns:
    score_df = pois_df.groupby("sa2_name").size().rename("poi_count").reset_index()
    score_df["z_poi"] = z_score(score_df["poi_count"])
    score_df["score"] = sigmoid(score_df["z_poi"])
else:
    score_df = pd.DataFrame(columns=["sa2_name", "poi_count", "z_poi", "score"])

score_df.head()

## Scoring Explanation Notes

Use this section to explain:

- Why POI count is a reasonable proxy for resource availability
- Why z-score normalisation is used
- Why sigmoid scaling is used
- Whether population below 100 was excluded
- Any extensions to the formula, such as population adjustment or POI category weighting

# Task 4: Report Analysis and Visualisation

## 4.1 Key Findings from Task 1

Write the main statistical findings here after completing the derived statistics.

Possible angles:

- Population change over time
- Age structure
- Gender differences
- Density or growth indicators
- Measures with unusual changes or missingness

## 4.2 Score Visualisation Plan

Add plots here once `score_df` is populated.

Recommended visuals:

- Histogram of SA2 scores
- Top and bottom ranked SA2 regions
- Map overlay or choropleth if boundary geometry is available
- POI category breakdown by SA4 or SA2

In [ ]:
if not score_df.empty:
    display(score_df.sort_values("score", ascending=False).head(10))
    display(score_df.sort_values("score", ascending=True).head(10))
    display(score_df["score"].describe())
else:
    print("Score table is empty. Complete Task 2 before generating score summaries.")

## 4.3 Limitations

Discuss the limitations of the analysis here.

Possible limitations:

- POI count does not measure service quality or capacity
- Larger SA2s may naturally contain more POIs
- Population size may need to be considered
- API completeness and category definitions may affect results
- Bounding boxes can include POIs outside the actual SA2 polygon unless geometry filtering is added

# Next Steps

1. Add group member names and selected SA4 zones.
2. Complete Task 1 derived statistics.
3. Add SA2 boundary data for selected SA4 zones.
4. Confirm the NSW Points of Interest API endpoint from the Week 8 tutorial.
5. Store POI data in the local SQLite database.
6. Generate score visualisations and write report findings.